 (896x21504 and 6144x128)

MTS_args: Namespace(is_training=True, model_id='336_24', model='SVQ', data='custom', data_name='ETTh1', root_path='../../../dataset/ETT-small/', data_path='ETTh1.csv', features='M', target='OT', freq='h', checkpoints='./checkpoints/', pretrain_checkpoints='../../../pretrain_checkpoints/', diffusion_config_dir='../../../model9_NS_transformer/configs/toy_8gauss.yml', seq_len=336, label_len=48, pred_len=24, enc_in=7, dec_in=7, c_out=7, fc_dropout=0.2, head_dropout=0.0, p_hidden_dims=[64, 64], p_hidden_layers=2, wFFN=0, svq=1, codebook_size=256, sout=0, individual=0, revin=1, affine=0, subtract_last=0, length=96, num_codebook=1, d_model_c=512, n_heads_c=8, e_layers_c=2, d_layers_c=1, d_ff=512, moving_avg=25, factor_c=3, distil=True, dropout=0.2, embed='timeF', activation='gelu', output_attention=False, do_predict=False, denoise_model='PatchDN', patch_size=16, stride=8, padding_patch='end', depth=1, d_model_d=128, n_heads_d=8, ddpm_inp_embed=256, ddpm_dim_diff_steps=256, ddpm_channels_conv=256, ddpm_channels_fusion_I=256, ddpm_layers_inp=5, ddpm_layers_I=5, ddpm_layers_II=5, cond_ddpm_num_layers=5, cond_ddpm_channels_conv=64, decomposition=False, kernel_size=15, fourier_factor=1.0, num_workers=4, itr=1, train_epochs=100, batch_size=128, test_batch_size=1, patience=15, learning_rate=0.0001, des='Exp', loss='mse', lradj='type1', use_amp=False, use_gpu=True, gpu=0, use_multi_gpu=False, devices='0,1,2,3', seed=2021, timesteps=100, sampling_timesteps=50, fastsample=False, type_sampler='DPM_solver', DPMsolver_step=20, eta=1, parameterization='x_start', bias=True, use_pretraining_condition=False, cond_pred_model_requires_grad=False, from_scrach=True, scrach_10_stop=True, bias_y_0=False)

Place the checkpoint of the already trained point prediction model into this folder.
path: pretrain_checkpoints/SVQ/all/weather/192/checkpoint.pth

In [ ]:
from data_provider.data_loader import Dataset_ETT_hour, Dataset_ETT_minute, Dataset_Custom, Dataset_Pred
from torch.utils.data import DataLoader

data_dict = {
    'ETTh1': Dataset_ETT_hour,
    'ETTh2': Dataset_ETT_hour,
    'ETTm1': Dataset_ETT_minute,
    'ETTm2': Dataset_ETT_minute,
    'custom': Dataset_Custom,
}


def data_provider(args, flag):
    Data = data_dict[args.data]
    timeenc = 0 if args.embed != 'timeF' else 1

    if flag == 'test':
        shuffle_flag = False
        drop_last = False
        batch_size = 1
        freq = args.freq
    elif flag == 'pred':
        shuffle_flag = False
        drop_last = False
        batch_size = 1
        freq = args.freq
        Data = Dataset_Pred
    else:
        shuffle_flag = True
        drop_last = True
        batch_size = args.batch_size
        freq = args.freq

    data_set = Data(
        root_path=args.root_path,
        data_path=args.data_path,
        flag=flag,
        size=[args.seq_len, args.label_len, args.pred_len],
        features=args.features,
        target=args.target,
        timeenc=timeenc,
        freq=freq
    )
    print(flag, len(data_set))
    data_loader = DataLoader(
        data_set,
        batch_size=batch_size,
        shuffle=shuffle_flag,
        num_workers=args.num_workers,
        drop_last=drop_last)
    # ✅ 对于 test，仅返回最后一个样本
    if flag == 'test':
        last_sample = list(data_loader)[-1:]
        return data_set, last_sample
    return data_set, data_loader
